# 4. SAVI and temporal-feature preparation

Load selected, partitioned four-month windows and build three comparable feature designs. This notebook does not select new windows or download imagery.

**Run order:** [Four-month windows](03_four_month_windows.ipynb) → this notebook → [Model training](05_model_training.ipynb).

All three designs use the same pixel rows and partitions. Automatic window selection, crop classes, source comparisons and the 500-pixel cap remain unchanged. Inputs must use the updated bilinear B5/B11 imagery contract.

**Active run: `bilinear_expanded`.** Run after the window notebook. Formal feature bundles are written inside this run only. Pending review or freeze status remains visible; do not mark a scientific decision frozen just to make a handoff pass. Local exploratory training in the next notebook builds the same three feature designs directly from complete pixel tables, so it does not require a prepared formal handoff.


## 1. Set paths and freeze the feature recipe

Keep `feature_policy_frozen = False` while developing the recipe. Before training, record your development decision and set it to `True`; this does not override the separate frozen window/training configuration.

SAVI uses scaled reflectance and soil adjustment L=0.5 by default. This is an explicit development choice. The formula is `(1 + L) × (B8 − B4) / (B8 + B4 + L)`.

Output: `Window inputs: <path>` — selected, partitioned windows exported by Step 14.

Output: `Feature freeze: <value>` — whether the feature recipe is ready for fitting.


In [ ]:
from pathlib import Path
import json
import os
from crop_pipeline.features import experiments as feature_experiments

# Notebooks live in notebooks/; run from the repository root so relative paths resolve.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
settings_path = Path("settings.local.json")
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
report_folder = (
    Path(local_settings.get("report_root", "./sen4agrinet_work/inventory")).expanduser().resolve()
)
analysis_run_name = "bilinear_expanded"  # Keep the previous pilot results separate.
season_folder = report_folder.parent / "analysis" / analysis_run_name / "seasons"
feature_policy = dict(feature_experiments.FEATURE_POLICY)
feature_policy_frozen = False
feature_development_evidence = ""
print("Window inputs:", season_folder)
print("Feature freeze:", feature_policy_frozen)


## 2. Inspect the three feature designs

| Design | Predictors | Count |
|---|---|---|
| `interval_only` | Five bands and NDVI, NDRE, NDMI, SAVI in each of eight bins | 72 |
| `interval_and_shape` | Interval predictors plus temporal-shape predictors | 120 |
| `shape_only` | Temporal-shape predictors for the four indices | 48 |

Each index contributes seven consecutive differences, green-up and senescence slopes, first peak interval, amplitude and trapezoidal integral. Slopes use actual calendar days. The integral covers first through last bin midpoint; no values are extrapolated outside those midpoints.

A peak at a boundary leaves one phase unobserved. Its phase slope is defined as zero and a separate metadata flag records that absence. These flags are not classifier inputs. Tied peaks use the first maximum.

Output: `<design> <number>` — ordered predictor count for that experiment.


In [ ]:
for design_name, columns in feature_experiments.feature_sets().items():
    print(design_name, len(columns))


## 3. Export separate primary and exploratory feature handoffs

The primary and exploratory window policies stay separate. Source row identities, labels, timing evidence and partitions are preserved. Undefined SAVI or incomplete data raises an error rather than silently changing one experiment's samples.

Output: `<bundle> <status> rows: <number>` — rows exported; pending means input rows or either development freeze is missing.

Output: `Feature handoff: <path>` — hash-verified input for the training notebook. Each bundle contains features, configuration, coverage and a window-input provenance snapshot. Keep its files together.


In [ ]:
feature_handoffs = {}
for window_folder_name, feature_folder_name in [
    ("training_input", "feature_input"),
    ("exploratory_training_input", "exploratory_feature_input"),
]:
    source_handoff = season_folder / window_folder_name / "training_handoff.json"
    destination = season_folder / feature_folder_name
    feature_handoff = feature_experiments.export_feature_handoff(
        source_handoff,
        destination,
        policy=feature_policy,
        frozen=feature_policy_frozen,
        development_evidence=feature_development_evidence,
    )
    feature_rows, configuration, coverage, manifest = feature_experiments.load_feature_handoff(
        feature_handoff
    )
    feature_handoffs[feature_folder_name] = feature_handoff
    print(feature_folder_name, manifest["status"], "rows:", len(feature_rows))
    print("Feature handoff:", feature_handoff)


## Continue to training

Open [05_model_training.ipynb](05_model_training.ipynb). The default local exploratory mode reads this run's complete window tables and computes the three designs internally. Pending formal feature handoffs do not block that path.

Production mode reads `feature_input/feature_handoff.json` or the explicitly selected exploratory feature bundle, and requires reviewed inputs and frozen settings. See [training](../docs/training.md).
